In [ ]:
import os
import re
from dotenv import load_dotenv
import pandas as pd
from google import genai
from google.genai import types
from pydantic import BaseModel, Field 
from huggingface_hub import login
from datasets import load_dataset
from transformers import AutoTokenizer
import matplotlib.pyplot as plt
import pickle
import concurrent.futures
import json
from queue import Queue
from openai import OpenAI
from dotenv import load_dotenv
import random

In [ ]:


def create_high_recall_snapshot(subject: str, body: str) -> str:
    """
    Slices a long email into a dense layout optimized to preserve urgency cues
    while respecting strict local VRAM and token limits.
    """
    
    clean_body = " ".join(body.split())
    clean_body.replace(">" , "\n")
    
    if len(clean_body) <= 1000:
        return f"Subj: {subject}\nBody: {clean_body}"
        
    # 1. Grab the Head (Highly likely to contain the core intent)
    head = clean_body[:700]
    
    # 2. Grab the Tail (Highly likely to contain deadlines/actions)
    tail = clean_body[-300:]
    
    # 3. Now we don't know what the middle of the message is like so lets use dynamic approach (selecting the portion with urgent looking words and intent)
    urgency_keywords = r"(?i)(asap|urgent|broken|fail|blocker|deadline|critical|override|postpone|cancel)"
    middle_context = ""
    
    # Scan the hidden middle portion of the email
    middle_text = clean_body[700:-300]
    matches = list(re.finditer(urgency_keywords, middle_text))
    
    if matches:
        # Extract a 100-character window around the first critical match found
        first_match = matches[0]
        start = max(0, first_match.start() - 40)
        end = min(len(middle_text), first_match.end() + 60)
        middle_context = f"\n[MID-SIGNAL]: ... {middle_text[start:end]} ..."

    # 4. Construct the dense snapshot
    dense_input = (
        f"Subj: {subject}\n"
        f"Body-Head: {head} ...{middle_context}\n"
        f"Body-Tail: ... {tail}"
    )
    return dense_input

In [ ]:
# data = load_dataset('Yale-LILY/aeslc')
df = pd.read_pickle('triage_dataset.pkl')

In [ ]:
# train = data['train']
df.describe()
# test = data['test']
# validation = data['validation']

In [ ]:
# print(train[0])
df = df[:14200]


In [ ]:
l = []
for i in train:
    l.append(len(i['email_body']))
count = [1 for i in l if i>200]
print(l[19])

In [ ]:
print(train[100]) # As we can see that we cant judge a urgency of an email by its size
# So lets take the minimin body size to be 100 and below that will take the 

In [ ]:
plt.plot(l)
plt.title("Email_Body length distribution.")
plt.show()

In [ ]:
BASEMODEL= "google/gemma-4-e4b"
MIN_CHAR=100
MAX_CHAR=1500


In [ ]:
load_dotenv(override=True)
API_KEY = 'ollama3'
print(f"Your api key is : {API_KEY[:7]}")
BASE_URL = "http://localhost:11434/v1"  
MODEL_NAME = "llama3:latest"

client = OpenAI(api_key = API_KEY, base_url = BASE_URL)

categories = ["Urgent", "Scheduling", "Info", "Social", "Others"]


sys_prom = f"""You are a data labeling engine for notification triaging.
Your task is to analyse raw messages (which can be unformed chat messages or unstructured emails) and you task is to generate the output:
1. intent_output: It MUST be a informative one line summary of the message.
2. Category: Must be exactly one word from this list: {categories}.

---
Example Input:
Subj: Your approval is requested
Body: I don't know Mog Hue. Please return this request to the person who originated it so that the Manager section can be completed. By including the manager's name, that will help me to identify who this person is, what job they perform and whether or not the request makes sense. I have over 400 people in my group, so without more detail on these requests it is not worth taking my time to approve these. --Sally


Example Output:
intent_output: Sally is rejecting an unidentifiable approval request and directing it back to the originator for missing manager details.
Category: Urgent
---

You must respond only in the following formate. Do not write Intros, Outros or explainations:
intent_output: <informative-one-line-summary>
Category: <one-word-category>"""




In [ ]:
def labeling(raw_message):
    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role" : "system", "content": sys_prom},
                {"role" : "user" , "content": f"Input_Message: \n{raw_message}"}
            ],
            temperature= 0.1,
            max_tokens= 150
            
        )
        output = response.choices[0].message.content.strip()
        intent_match = re.search(r"intent_output:\s*(.*)", output, re.IGNORECASE)
        category_match = re.search(r"Category:\s*(.*)", output, re.IGNORECASE)
        
        if intent_match:
            intents = intent_match.group(1).strip()
            
                
        if category_match:
            
            category = category_match.group(1).strip()
            for cat in categories:
                if cat.lower() in category.lower():
                    category = cat
                    break
        
        return {
                "message": raw_message,
                "intent_output": intents,
                "category": category,
            }
    except Exception as e:
        print(f"Error processing row: {e}")
    return None
        

In [ ]:
import re
from typing import Optional, Dict, Any
from transformers import AutoTokenizer

class TriageItem:
    # Use Gemma 4 / Llama 3.2 tokenizers cleanly
    tokenizer = AutoTokenizer.from_pretrained("google/gemma-4-e4b", trust_remote_code=True)
    
    # Structural triggers for the local model's rationale chain
    THOUGHT_OPEN = "<|channel>thought\n"
    THOUGHT_CLOSE = "\n<channel|>"
    
    def __init__(self, raw_data: Dict[str, Any], max_tokens: int = 512):
        self.sender: str = raw_data.get("sender", "Unknown")
        self.subject: str = raw_data.get("subject", "No Subject")
        self.raw_body: str = raw_data.get("body", "")
        self.intent: str = raw_data.get("intent", "Info")
        self.rationale: str = raw_data.get("rationale", "")
        
        self.max_tokens = max_tokens
        self.prompt: Optional[str] = None
        self.include: bool = False
        
        self.process()

    def clean_whitespace(self, text: str) -> str:
        """Removes layout noise without destroying critical words/codes."""
        if not text:
            return ""
        # Collapse multiple spaces/newlines but keep structural integrity
        text = re.sub(re.compile(r'\s+'), ' ', text)
        return text.strip()

    def process(self):
        """Processes raw text and builds a balanced token-budget constraint prompt."""
        cleaned_body = self.clean_whitespace(self.raw_body)
        
        # Build standard system/user entry structure
        user_input = f"From: {self.sender} | Subj: {self.subject}\nBody: {cleaned_body}"
        
        # Tokenize early to protect character boundaries
        input_tokens = self.tokenizer.encode(user_input, add_special_tokens=False)
        
        # Guardrail: Enforce physical token ceilings for fast local GPU pre-fill
        if len(input_tokens) > self.max_tokens:
            input_tokens = input_tokens[:self.max_tokens]
            user_input = self.tokenizer.decode(input_tokens, skip_special_tokens=True)
            
        # Structure the golden target response (What Gemma 4 must learn to output)
        target_output = (
            f"{self.THOUGHT_OPEN}{self.rationale}{self.THOUGHT_CLOSE}"
            f"INTENT: {self.intent}"
        )
        
        # Assemble the fine-tuning training pair
        self.prompt = f"Instruction: Triage the following notification snapshot.\n\nInput:\n{user_input}\n\nOutput:\n{target_output}"
        self.include = True

    def test_prompt(self) -> str:
        """Returns the prompt sliced exactly where the local agent must begin generation."""
        return self.prompt.split("Output:\n")[0] + "Output:\n"

    def __repr__(self) -> str:
        return f"<TriageItem From={self.sender} Intent={self.intent}>"

In [ ]:
# train_new = [create_high_recall_snapshot(i['subject_line'], i["email_body"]) for i in train]
train_new = [
    f"{df.loc[i, 'message']}\n[[[[[INTENT]]]]]:{df.loc[i, 'intent_output']}"
    for i in df.index
]


In [ ]:
print(train_new[1000].split('[[[[[INTENT]]]]]:'))

In [ ]:
CHUNK_SIZE = 500
OUTPUT_FILE = 'Triage_dataset_Next_3.pkl'
def process_batch(batch, executor):
    futures = {executor.submit(labeling, msg): msg for msg in batch}
    results = []
    for future in concurrent.futures.as_completed(futures):
        try:
            res = future.result()
            if res:
                results.append(res)
        except Exception as e:
            print(f"Error: {e}")
    return results

def stream_process(messages):
    generated = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
        for i in range(0, len(messages), CHUNK_SIZE):
            batch = messages[i:i+CHUNK_SIZE]
            print(f"Processing batch {i//CHUNK_SIZE+1} ({len(batch)} items)...")
            batch_results = process_batch(batch, executor)
            generated.extend(batch_results)

            # Save progress incrementally
            df = pd.DataFrame(generated)
            with open(OUTPUT_FILE, "wb") as f:
                pickle.dump(df, f, protocol=pickle.HIGHEST_PROTOCOL)

            print(f"Saved {len(generated)} items so far...")

    return generated



In [ ]:
import pickle
import pandas as pd

# Define the exact same file name you used to save it
OUTPUT_FILE = "Triage_dataset_Next_2.pkl"

try:
    # Open and load the pickle file
    with open(OUTPUT_FILE, "rb") as f:
        df = pickle.load(f)
    
    # Print the total number of processed rows
    print(f"📊 Total Rows Processed successfully: {len(df)}")
    print("-" * 50)
    
    # Show the first 5 rows to check the content
    print("👀 First 5 rows of your dataset:")
    print(df.head())
    
    print("-" * 50)
    # Check the distribution of your categories to see how Llama classified them
    print("🗂️ Category Distribution Breakdown:")
    print(df['category'].value_counts())

except FileNotFoundError:
    print(f"❌ Error: Could not find '{OUTPUT_FILE}' in this directory. Make sure the path matches.")
except Exception as e:
    print(f"❌ An error occurred while loading the data: {e}")

In [ ]:
df = pd.read_pickle('Triage_dataset_Next.pkl')

In [ ]:
sample = []
for i in range(0, 400):
    if df['category'][i] == 'Social':
        sample.append((df['message'][i],
        df['intent_output'][i],
        df['category'][i]))
    if len(sample) == 50:
        break

In [ ]:
sample

In [ ]:
import random
def generate(sample):
    temp = random.choice(sample)
    sys_prom = f"""
    You are an advance data generating engine for private-triage. Your tast is to generate completely new high quality synthetic data with the help of the data provided.
    You MUST match the exact formatting of the reference example provided below. Output ONLY a valid Python string.
    EXAMPLE:
    -------------------------------------------------------------------------------------

    message: \n{temp[0]}
    Intent {temp[1]}
    category:  {temp[2]}

    -------------------------------------------------------------------------------------

    ### CRITICAL RULES:
    1. Create a completely brand-new situation, name, and topic. Do not repeat the reference example.
    2. Maintain the raw email realism (include Subj: and Body: tags, keep realistic human typos or shorthand if the reference has them).
    3. The category must be exactly 'Social'.
    4. Output NOTHING except the raw tuple string: ('Subj: ...\\nBody: ...', ':One line Intent...', 'Social')
    this is just the basic formate try to think above it and create a structured brand new social emaildata without any repetation.
    """
    return sys_prom
    


In [ ]:
print(generate(sample))

In [ ]:
Extra_data = pd.read_csv('emails.csv')

In [ ]:
print(len(Extra_data))

In [ ]:
messages = []
for i in Extra_data['message']:
    messages.append(i)

In [ ]:
print(messages[1])

In [ ]:


def parse_extra_data(data):  
    
    match = re.search(r"Subject:\s*(.*)", data)
    
    
    if not match or not match.group(1).strip():
        
        if "Mime-Version:" in data:
            return None
        subject_raw = ""
        
    else:
        subject_raw = match.group(1).strip()
    
    
    if subject_raw.startswith("Mime-Version:"):
        return None
        
    if subject_raw.startswith("Re:"):
        # Strip 'Re:' and any trailing spaces from the start
        subject = re.sub(r"^Re:\s*", "", subject_raw)
        if subject.startswith("Mime-Version:"):
            return None
    else:
        subject = subject_raw

    parts = data.split('\n\n')
    if len(parts) < 2:
        return None  # No body content present
        
    body = '\n'.join(parts[1:])
    cleaned = re.sub(r'[^A-Za-z0-9\s.,?!]', '', body)
    
   
    if cleaned.strip().startswith('Forwarded') or len(cleaned) <100:
        return 
    if not subject or not cleaned:
        return 


    return create_high_recall_snapshot(subject, cleaned)

In [ ]:
messages[0]

In [ ]:
new_messages = []
for i in messages:
    k = parse_extra_data(i)
    if k:
        new_messages.append(k.split('\n\n')[0])
        



In [ ]:
print(new_messages[0])
len(new_messages)

In [ ]:
import random
print(random.choice(new_messages))

In [ ]:
random.seed(42)
t = labeling(new_messages[0])
print(t['message'])
print(t['intent_output'])
print(t['category'])

In [ ]:
t = [len(i) for i in new_messages]
plt.plot(t)
plt.show()

In [ ]:
seen = set()
unlabeled = []
for i in range(20*len(new_messages)):
    k = random.choice(new_messages)
    if k in seen:
        continue
    else:
        unlabeled.append(k)
        seen.add(k)

In [ ]:
len(unlabeled)

In [ ]:
if __name__ == "__main__":
    # Replace with your dataset
    labeled = unlabeled[14000 : 20000]
    print(f"Streaming {len(labeled)} items in chunks of {CHUNK_SIZE}...")
    final_results = stream_process(labeled)
    print("✅ All done! Final dataset ready for Hugging Face deployment.")


In [ ]:
df = pd.read_pickle('Triage_dataset_Next.pkl')

In [ ]:
df.describe()

In [ ]:
df2 = pd.read_pickle('Triage_dataset_Next_2.pkl')
df3 = pd.read_pickle('Triage_dataset_Next_3.pkl')

In [ ]:
df = df 

In [ ]:
print(df['category'].value_counts())

In [ ]:
len(Info)

In [ ]:
Info = [
    [row['message'], row['intent_output'], row['category']]
    for d in [df, df2, df3]
    for _, row in d.iterrows()
    if row['category'] == 'Info'
]
Scheduling = [
    [row['message'], row['intent_output'], row['category']]
    for d in [df, df2, df3]
    for _, row in d.iterrows()
    if row['category'] == 'Scheduling'
]

Urgent = [
    [row['message'], row['intent_output'], row['category']]
    for d in [df, df2, df3]
    for _, row in d.iterrows()
    if row['category'] == 'Urgent'
]

Social = [
    [row['message'], row['intent_output'], row['category']]
    for d in [df, df2, df3]
    for _, row in d.iterrows()
    if row['category'] == 'Social'
]


In [ ]:
Info[0]

In [ ]:
# with no head
new_info = []
for i in Info:
    k = i[0].split('Body-Head:')
    if len(k) == 1:
        new_info.append(i) 
len(new_info)
new_scheduling = []
for i in Scheduling:
    k = i[0].split('Body-Head:')
    if len(k) == 1:
        new_scheduling.append(i)
len(new_scheduling) 
new_social = []
for i in Social:
    k = i[0].split('Body-Head:')
    if len(k) == 1:
        new_social.append(i)
len(new_social)
new_urgent = []
for i in Urgent:
    k = i[0].split("Body-Head:")
    if len(k) == 1:
        new_urgent.append(i)
len(new_info)
    

In [ ]:
Final = new_info + new_scheduling + new_social + new_urgent
random.shuffle(Final)

In [ ]:
del data_final
data_final = pd.DataFrame(columns=["message", "intent_output"])

In [ ]:
for i in Final:
    k = i[:2]
    data_final.loc[len(data_final)] = k

In [ ]:
data_final.head(10)

In [ ]:
data_final.to_pickle("Final_training_set")

In [ ]:
len(Final)

In [ ]:
for i in range(len(Final)):
    if Final[i][1][0] == ':':
        Final[i][1] = Final[i][1][1:]
    if "LOG MESSAGES" in Final[i][1]:
        continue
    else:
        k = Final[i][:2]
        data_final.loc[len(data_final)] = k

In [ ]:
data_final.head(11)

In [71]:
data_final.describe()


,message,intent_output
count,23439,23439
unique,22831,22428
top,Subj: EFCU Gets You Connected\nBody: Get Conne...,Notification of final scheduling ISO file pars...
freq,15,34


In [73]:
data_final['message'].drop_duplicates()

0        Subj: Rodeo Tickets\nBody: Are any of these av...
1        Subj: CA Development\nBody: Attached are elect...
2        Subj: Mike/terry/John\nBody: CALENDAR ENTRY AP...
3        Subj: Credit Watch List--9/25/00\nBody: Attach...
4        Subj: EPE SCHEDULE AT 4C345 - HLH -DELAYED TAG...
                               ...                        
23434    Subj: WARNING:  Your mailbox is approaching th...
23435    Subj: Industry Standard's Net Returns speakers...
23436    Subj: RE: LNG Changes for last week\nBody: com...
23437    Subj: FW: Another photo spot\nBody: Photo of m...
23438    Subj: Janet Wallis' Reliant Entex and Entex Ga...
Name: message, Length: 22831, dtype: str

In [74]:
data_final['intent_output'].drop_duplicates()

0        Request for rodeo tickets with specific perfor...
1        Confirm the number of original documents requi...
2        Appointment scheduled for Mike, Terry, and Joh...
3        A revised credit watch list is being distribut...
4        Confirm delayed tags at 4C345 due to miscommun...
                               ...                        
23433    Approve or deny counterparty access and update...
23435    Jeff is being asked to consider speaking at an...
23436    John Arnold is asked to provide tutoring and m...
23437    User shares a personal photo link with instruc...
23438    Janet Wallis requests confirmation on unsigned...
Name: intent_output, Length: 22428, dtype: str

In [75]:
data_final.describe()

,message,intent_output
count,23439,23439
unique,22831,22428
top,Subj: EFCU Gets You Connected\nBody: Get Conne...,Notification of final scheduling ISO file pars...
freq,15,34


In [79]:
data_final['message'] = data_final['message'].astype(str).str.strip().str.replace(r'\s+', ' ', regex=True)
data_final['intent_output'] = data_final['intent_output'].astype(str).str.strip().str.replace(r'\s+', ' ', regex=True)

# 2. Explicitly drop duplicates targeting ONLY the 'text' column
# This forces Pandas to look at the message content, ignoring minor variations in summary text.
data_final = data_final.drop_duplicates(subset=['message'], keep='first').copy()

# 3. Verify the fix worked perfectly
print("--- Verification Check ---")
print(f"Total Rows Before: {len(data_final)}")
print(f"Total Rows After:  {len(data_final)}")
print(f"Unique Texts Count: {data_final['message'].nunique()}")

# Look at the top frequencies again—they should all be exactly 1 now!
print("\nNew Frequency Check for Text:")
print(data_final['message'].value_counts().head(3))

--- Verification Check ---
Total Rows Before: 22756
Total Rows After:  22756
Unique Texts Count: 22756

New Frequency Check for Text:
message
Subj: Rodeo Tickets Body: Are any of these available? Forwarded by Kay MannCorpEnron on 01232001 0211 PM Carolyn George 01232001 0208 PM To Kay MannCorpEnronEnron cc Subject Rodeo Tickets The performances I am interested in are Clay Walker Destinys Child Leanne Womak Thanks, Carolyn George Sr. Administrative Assistant Enron Wholesale Services Enron Industrial Markets LLC 1400 Smith, EB3836A Houston, TX 77002 713 8533439 Fax 713 6463393 email carolyn.georgeenron.com    1
Subj: CA Development Body: Attached are electronic versions of the final docs for the CA Development transactions. If there are no other comments we will get the Enron signatures. How many originals are needed? Noticed was faxed to the Admin Agent this am. Please let me know if you have any questions. Kay                                                                             

In [81]:
data_final.head(100)

,message,intent_output
0,Subj: Rodeo Tickets Body: Are any of these ava...,Request for rodeo tickets with specific perfor...
1,Subj: CA Development Body: Attached are electr...,Confirm the number of original documents requi...
2,Subj: Mike/terry/John Body: CALENDAR ENTRY APP...,"Appointment scheduled for Mike, Terry, and Joh..."
3,Subj: Credit Watch List--9/25/00 Body: Attache...,A revised credit watch list is being distribut...
4,Subj: EPE SCHEDULE AT 4C345 - HLH -DELAYED TAG...,Confirm delayed tags at 4C345 due to miscommun...
...,...,...
95,Subj: Updated: Wholesale Business Plan Body: W...,Meeting invitation for discussing Wholesale Bu...
96,Subj: RE: Robertson Stephans Body: Of course n...,Sara is asking Jason for an update on the prio...
97,"Subj: (no subject) Body: Jana, Cotton Mary sou...","Vince and Jana discuss movie recommendations, ..."
98,Subj: Counterparties Body: Thank you all for b...,Request to confirm receipt of claims from list...


In [82]:
data_final.to_pickle("Final_training_set_with_removed_category.pkl")